In [2]:
import pandas as pd

In [3]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [ ]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit

# ==========================================
# 1. FEATURE ARCHITECTURE (The Alpha Factory)
# ==========================================
def build_engine_features(df):
    """
    Transforms raw input into Physical Alphas.
    Logic: Lags capture memory, Delta captures velocity, Accel captures force.
    """
    df = df.sort_values(['Id']).copy() # Ensure temporal order
    
    # Identify raw features (f0...fN)
    raw_cols = [c for c in df.columns if c.startswith('f')]
    
    # A. Momentum & Velocity (f0 as the primary flow indicator)
    df['f0_delta'] = df['f0'].diff().fillna(0)
    
    # B. The Temporal Memory Stack (f20 as the liquidity indicator)
    for i in range(1, 6):
        df[f'f20_lag_{i}'] = df['f20'].shift(i).fillna(0)
        
    # C. Kinetic Energy & Acceleration
    df['f20_energy'] = df['f20']**2
    df['f20_accel'] = df['f20'].diff().diff().fillna(0)
    
    return df

# ==========================================
# 2. SYSTEM TRAINING (Signal + Risk Layers)
# ==========================================
print("📡 Step 1: Alpha Engineering on Training Data...")
train_proc = build_engine_features(train_df)
FEATURES = [c for c in train_proc.columns if c.startswith('f')]

print("📡 Step 2: Training Signal Experts (Forward-Chained)...")
tscv = TimeSeriesSplit(n_splits=5)
oof_signal_preds = np.zeros(len(train_proc))
signal_models = []

for train_idx, val_idx in tscv.split(train_proc):
    X_t, y_t = train_proc.iloc[train_idx][FEATURES], train_proc.iloc[train_idx]['y']
    X_v, y_v = train_proc.iloc[val_idx][FEATURES], train_proc.iloc[val_idx]['y']
    
    # Huber Loss for robustness against black-swan spikes
    m = LGBMRegressor(objective='huber', n_estimators=400, learning_rate=0.03, reg_alpha=10)
    m.fit(X_t, y_t)
    
    oof_signal_preds[val_idx] = m.predict(X_v)
    signal_models.append(m)

print("🛡️ Step 3: Training Meta-Risk Layer (Information Theory)...")
# Isolate valid OOF predictions to train the seatbelt
valid_idx = np.where(oof_signal_preds != 0)[0]
# Target: Log-Absolute-Error (Stabilizes variance for learning)
meta_target = np.log1p(np.abs(train_proc.iloc[valid_idx]['y'] - oof_signal_preds[valid_idx]))

model_risk = LGBMRegressor(n_estimators=300, learning_rate=0.05)
model_risk.fit(train_proc.iloc[valid_idx][FEATURES], meta_target)

# CALIBRATION: Determine the threshold of "Normal" Market Behavior
risk_train_preds = model_risk.predict(train_proc.iloc[valid_idx][FEATURES])
risk_mu, risk_std = np.mean(risk_train_preds), np.std(risk_train_preds)
SAFE_LIMIT = np.percentile(np.abs(oof_signal_preds[valid_idx]), 99)

# ==========================================
# 3. PRODUCTION INFERENCE (Memory-Safe)
# ==========================================
def production_execution(test_raw, chunk_size=200_000):
    """
    Executes feature engineering and dual-layer prediction batch-wise.
    """
    n = len(test_raw)
    final_predictions = np.zeros(n)
    
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        # Apply Alphas to chunk
        chunk = build_engine_features(test_raw.iloc[start:end])
        X = chunk[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
        
        # A. Signal Ensemble
        raw_sig = np.mean([m.predict(X) for m in signal_models], axis=0)
        
        # B. Risk Gating (The Seatbelt)
        p_risk = model_risk.predict(X)
        z = (p_risk - risk_mu) / (risk_std + 1e-6) - 1.2 # -1.2 is the Gas Pedal
        confidence = 1 / (1 + np.exp(z)) # Sigmoid activation
        
        # C. Aggregation & Constraint
        # We multiply signal by confidence to mute noisy regimes
        final_predictions[start:end] = np.clip(raw_sig * confidence, -SAFE_LIMIT, SAFE_LIMIT)
        
    return final_predictions

# ==========================================
# 4. FINAL SUBMISSION
# ==========================================
print(" Step 4: Executing Final Inference...")
final_y = production_execution(test_df)

submission = pd.DataFrame({
    'Id': test_df['Id'],
    'y': final_y
})

submission.to_csv('submission.csv', index=False)
print(f"🏁 DONE. Generated {len(submission)} predictions.")

📡 Step 1: Alpha Engineering on Training Data...
📡 Step 2: Training Signal Experts (Forward-Chained)...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.045824 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8437
[LightGBM] [Info] Number of data points in the train set: 336192, number of used features: 34
[LightGBM] [Info] Start training from score -0.000009
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warn